# Battery Digital Twin — PINN Training (Revised v8)

This notebook rebuilds the original Databricks PINN battery digital twin pipeline, with sixteen fixes applied:

1. **Cell 11 (feature engineering):** logs how many cycles are skipped and why, instead of silently dropping them.
2. **OCV model:** replaced the crude linear `U_ocv = 2.7 + 0.015 * SoC` with a fitted empirical curve that captures the nonlinear knees near 0% and 100% SoC.
3. **`log_A_sei` persistence:** the learned Arrhenius pre-exponential factor is now saved *with* the network checkpoint and restored at inference time, instead of resetting to its untrained default.
4. **Inference-time architecture reconstruction:** reuses the real physics PDE (instead of a `dummy_pde` placeholder) when rebuilding the network for weight loading, with an explicit note on what `model.predict()` does and does not enforce.
5. **GPU-native interpolation:** `get_real_time_physics` no longer round-trips through `scipy.interp1d` on the CPU for every PDE residual call — it now uses a `torch.searchsorted`-based linear interpolation that stays entirely on-device.
6. **Ambient temperature as a model input:** `ambient_temperature` (previously dropped before `final_pinn_training_data`) is now carried through the pipeline and added as a 5th input dimension `[space, SoH, Re, Ambient, time]`. `T_delta` is re-referenced per-battery against its own ambient condition instead of a single global `T_ref = 25°C` — this addresses the diverging PDE residual losses (`loss_comp_0`/`loss_comp_1` climbing over training) observed in the MLflow run, which is consistent with a single global thermal reference being physically inconsistent across the dataset's ~4°C / 24°C / 43°C ambient test groups.
7. **Rebalanced loss weights & fixed metric callback:** the data-fit term previously dominated the PDE residual terms 10–100x over, which is consistent with `log_A_sei` never moving from its untrained default and the PDE losses diverging. Loss weights are rebalanced to give the physics terms more relative weight, and the MLflow `MAE_Kelvin_Scaled`/`L2_Relative_Error` metrics — which were flatlined at 0.000 because `train_state.metrics_test` was mostly evaluated on unlabeled PDE points — are now computed directly against the real temperature observations.
8. **Bonus fix — oven abuse test now actually uses the temperature sweep:** the original abuse-test cell computed an ambient temperature ramp (`oven_t_c`, 25→150°C) but never fed it into the model — predictions were made along the ordinary time domain only, so the sweep was dead code. Now that ambient is a real input dimension, the sweep drives the abuse simulation directly.
9. **`A_sei` numerical/gradient-signal fix:** the MLflow run showed `A_sei_value` frozen dead flat at its untrained default for 15,000 steps. Diagnosed root cause: with `Ea_sei = 1.35e5` J/mol, the Arrhenius exponent is around −51 to −54 across the dataset's 0–45°C range — past the old `clamp(min=-50)` guard, making the reaction-rate term ~12+ orders of magnitude smaller than the Joule-heating term it's added to, so its gradient contribution was negligible regardless of loss weighting. The exponent clamp is widened to (−70, 50), and `log_A_sei`'s starting value is now computed diagnostically (targeting rate ≈ O(1) at a representative operating temperature) instead of an arbitrary guess. **Confirmed working:** the actual training run calibrated `A_sei` to 3.44×10²² (`log10 ≈ 22.54`), close to the diagnostic-recommended starting point of 22.88 — a real, moving, converging value instead of a frozen default.
10. **`ic_T` plateau — rebalanced further + added a diagnostic.** The confirmed training run showed `loss_comp_2` (`ic_T`) plateau flat at ~0.849 through the second half of Adam and all of L-BFGS, instead of continuing to decrease — a sign the data-fit term (`observe_T`, weight 20) may be winning a conflict against the `t=0` initial condition rather than both being satisfied. `ic_T`/`ic_c` weights are raised from 5 to 30, and a diagnostic cell after training checks whether `bin=0` (the first downsampled point per cycle) already shows measurable warm-up relative to ambient — which would mean some `ic_T` residual is a genuine data-resolution limit rather than a pure optimization failure.
11. **Medallion ETL (Section 2) marked and gated as one-off.** Bronze/Silver/Gold table creation is now controlled by a `RUN_MEDALLION_ETL` flag (default `False`), so it's skipped on every run after the first instead of needing to be commented out manually. Downstream sections only ever read from the persisted `gold_battery_pinn_database`/`final_pinn_training_data` tables.
12. **Digital Twin Restoration (Section 7) made self-contained, with a `run_id` placeholder.** It no longer silently depends on Sections 4–6 having just executed in the same kernel session — it now redefines every constant, interpolation table, and helper function it needs by re-reading the persisted `final_pinn_training_data` table, so it can run standalone after a kernel restart (as long as the one-off tables already exist). `run_id` is a clearly-marked placeholder that fails fast with a helpful error if left unset, and a missing/legacy checkpoint format now raises an explicit error instead of silently proceeding with wrong physics.
13. **Critical unit bug fixed: `Temperature_measured` is in Celsius, not Kelvin.** The v5 `ic_T` diagnostic (fix #10) surfaced this directly — it printed a bin-0 `(Temperature_measured - ambient)` averaging **≈ −272 K**, which is only possible if one side of that subtraction was in Celsius and the other in Kelvin. The previous `obs_values` formula added `+273.15` to `ambient_temperature` but left `Temperature_measured` unconverted, implicitly treating it as already being in Kelvin. Since NASA's `Temperature_measured` column is in Celsius — same as `ambient_temperature` — this is a pure difference and needs **no** Kelvin conversion at all (the `+273.15` on each side would cancel out algebraically even if both were correctly converted). Fixed in both `obs_values` (Section 5) and the bin-0 diagnostic (Section 6). **`sei_thermal_pde`'s own Arrhenius calculation was already correct** — it only converts the ambient *input* to Kelvin internally and never touches `Temperature_measured` directly, so no PDE code changes were needed, only the training-target formula that `T_delta` is compared against. This bug likely explains a substantial part of the earlier runs' odd behavior (e.g. the `ic_T` plateau, and `T_safe` frequently hitting its clamp floor), since the network was previously being trained against a target offset by roughly −273°C. **Confirmed fixed:** the next training run's bin-0 diagnostic read a sane `mean=0.80°C` instead of `-272K`, and `ic_T` dropped from a 0.849 plateau to `0.00235`.
14. **Abuse-test extrapolation warning.** The Section 7 oven abuse-test sweeps `Ambient` from 25°C to 150°C, but the network's `Ambient` input was only ever trained on `[0, 45]°C` (Section 5's `Hypercube(..., xmax=[..., 45.0])`, matching the dataset's real ~4/24/43°C test groups). Everything past 45°C ambient is the network extrapolating outside its trained domain, which is consistent with the previously-observed non-monotonic SEI-stability curve at high stress (chemical stability appearing to *recover* past ~165°C — a sign of an ungrounded extrapolation, not real chemistry). The plot now visually separates in-domain (solid) from extrapolated (dashed) predictions, prints an explicit warning when the sweep exceeds the trained domain, and restricts the SAFE/WARNING/DANGER status logic to only the in-domain portion of the sweep.
15. **Data-driven danger threshold.** The SEI-stability "chemical failure limit" was a hardcoded `0.985` with no clear grounding. **v8 note:** the first attempt at a data-driven threshold (1st percentile of the model's own in-domain `c` predictions) turned out to be methodologically circular — by definition ~1% of any distribution sits at or below its own 1st percentile, and the actual baseline showed `mean=1.0092`, above the physically valid `[0,1]` range, because the network's raw `c` output isn't clamped the way the internal `c_safe` used in the PDE residual is. Fixed by (1) clipping `c` to `[0,1]` before computing any statistics, and (2) using `mean − 3·std` of the clipped in-domain baseline as the threshold instead of a percentile of the same data — still a heuristic tied to this model's own behavior rather than a literature-validated failure point, but no longer tautological.
16. **DANGER trigger made auditable, and abuse-test data coverage checked.** The verdict previously reported only "DANGER" without saying which condition fired. `core_temp_in_domain.max()` and `sei_health_in_domain.min()` are now printed explicitly against their thresholds. A new diagnostic also checks how much real training data exists near the abuse test's fixed `SoH=0.8, Re=0.5` — satisfying the geometry's bounds doesn't guarantee real data density there, so a DANGER verdict at sparse conditions may reflect an undertrained region rather than real physics.

Sections:
- Setup
- Data Engineering (Bronze → Silver → Gold, Spark SQL)
- Feature Engineering (SoC / SoH / cycle downsampling)
- Physics Definition (PDE system, OCV fit, constants)
- Geometry & Boundary/Initial Conditions
- Model Architecture & Training
- Digital Twin Restoration & Abuse-Test Inference

**Note on `%pip install`:** `scipy` is still installed for the initial `polyfit`/setup convenience,
but is no longer required inside the training loop — the interpolation used during PDE residual
evaluation is now pure PyTorch.


## 1. Setup

In [0]:
# Installing PyTorch and DeepXDE
%pip install torch torchvision torchaudio
%pip install deepxde
%pip install scipy

# This restarts the Python process so it recognizes the new libraries
dbutils.library.restartPython()

In [0]:
import os
# Set the backend to pytorch
os.environ["DDE_BACKEND"] = "pytorch"

import torch
import deepxde as dde

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from tqdm import tqdm
import mlflow

# Verify installation
print(f"DeepXDE backend: {dde.backend.backend_name}")
print(f"PyTorch version: {torch.__version__}")

## 2. Data Engineering (Medallion Architecture) — ⚠️ ONE-OFF SETUP

Bronze -> Silver -> Gold, built with Spark SQL exactly as before. Battery IDs `B0049`–`B0052`
are excluded (kept consistent with the original notebook's exclusion list).

**This section only needs to run once** (or whenever the raw `metadata.csv`/discharge files
change) — it rebuilds `bronze_battery_metadata`, `silver_discharge_metadata`,
`silver_impedance_metadata`, and `gold_battery_pinn_database` from scratch, which is the slow,
one-off part of the pipeline. Section 3 onward only reads from `gold_battery_pinn_database`, so
once these tables exist in your workspace there's no need to rerun this section on every pass.

Set the flag below to control it:
- `RUN_MEDALLION_ETL = False` (default) — **skip** rebuilding Bronze/Silver/Gold and reuse the
  tables already in the workspace. Use this on every run after the first.
- `RUN_MEDALLION_ETL = True` — rebuild everything from the raw files. Use this only the first time,
  or after the raw data changes.

In [0]:
# ---------------------------------------------------------
#  One-off ETL toggle
# ---------------------------------------------------------
# Flip to True only when you need to (re)build Bronze/Silver/Gold from the raw
# files -- e.g. the very first run, or after the raw dataset changes. Leave as
# False on every subsequent run to skip straight to reusing the existing
# gold_battery_pinn_database table.
RUN_MEDALLION_ETL = False

if RUN_MEDALLION_ETL:
    print("RUN_MEDALLION_ETL=True -- rebuilding Bronze/Silver/Gold from raw files...")
else:
    print("RUN_MEDALLION_ETL=False -- skipping ETL, reusing existing gold_battery_pinn_database.")

In [0]:
if RUN_MEDALLION_ETL:
    # Load the raw files into Spark DataFrames
    raw_metadata = spark.read.csv("/Volumes/workspace/default/pinn/cleaned_battery_dataset/metadata.csv", header=True, inferSchema=True)

    # Register as Temporary Views so we can use SQL
    raw_metadata.createOrReplaceTempView("raw_metadata")

    # Save as a permanent Bronze Table
    raw_metadata.write.format("delta").mode("overwrite").saveAsTable("bronze_battery_metadata")

    display(spark.sql("SELECT * FROM raw_metadata LIMIT 50"))

In [0]:
if RUN_MEDALLION_ETL:
    spark.sql("""
    CREATE OR REPLACE TABLE silver_discharge_metadata AS
    WITH exploded_time AS (
      SELECT
        battery_id,
        type,
        filename,
        ambient_temperature,
        start_time,
        capacity,
        regexp_extract_all(start_time, r'(\\d+\\.?\\d*([eE][+-]?\\d+)?)') as parts
      FROM bronze_battery_metadata
      WHERE battery_id NOT IN ('B0049', 'B0050', 'B0051', 'B0052')
    ),
    formatted_time AS (
      SELECT *,
        CAST(CAST(parts[0] AS DOUBLE) AS INT) as y,
        CAST(CAST(parts[1] AS DOUBLE) AS INT) as m,
        CAST(CAST(parts[2] AS DOUBLE) AS INT) as d,
        CAST(CAST(parts[3] AS DOUBLE) AS INT) as hr,
        CAST(CAST(parts[4] AS DOUBLE) AS INT) as mi,
        CAST(parts[5] AS DOUBLE) as sec
      FROM exploded_time
    )
    SELECT
      battery_id, type, filename, ambient_temperature, capacity,
      to_timestamp(concat(y, '-', m, '-', d, ' ', hr, ':', mi, ':', sec), 'yyyy-M-d H:m:s.SSS') AS start_datetime,
      ROW_NUMBER() OVER(PARTITION BY battery_id ORDER BY y, m, d, hr, mi, sec) AS cycle_number
    FROM formatted_time
    WHERE type = 'discharge'
    """)

    display(spark.sql("""
        SELECT * FROM silver_discharge_metadata
        WHERE start_datetime IS NOT NULL
        LIMIT 50
    """))

In [0]:
if RUN_MEDALLION_ETL:
    spark.sql("""
    CREATE OR REPLACE TABLE silver_impedance_metadata AS
    WITH cleaned_strings AS (
      SELECT
        battery_id,
        Re,
        Rct,
        type,
        -- Strip brackets and split by one or more spaces
        split(trim(regexp_replace(start_time, r'\\[|\\]', '')), ' +') as parts
      FROM raw_metadata
      WHERE battery_id NOT IN ('B0049', 'B0050', 'B0051', 'B0052')
    ),
    formatted_time AS (
      SELECT
        *,
        -- Convert indices to numbers, handling potential scientific notation
        CAST(CAST(parts[0] AS DOUBLE) AS INT) as y,
        CAST(CAST(parts[1] AS DOUBLE) AS INT) as m,
        CAST(CAST(parts[2] AS DOUBLE) AS INT) as d,
        CAST(CAST(parts[3] AS DOUBLE) AS INT) as hr,
        CAST(CAST(parts[4] AS DOUBLE) AS INT) as mi,
        CAST(parts[5] AS DOUBLE) as sec
      FROM cleaned_strings
      -- Ensure we actually have 6 parts (Y, M, D, H, M, S)
      WHERE size(parts) >= 6
    )
    SELECT
      battery_id,
      Re,
      Rct,
      to_timestamp(
        concat(y, '-', m, '-', d, ' ', hr, ':', mi, ':', sec),
        'yyyy-M-d H:m:s.SSS'
      ) AS start_datetime,
      ROW_NUMBER() OVER(PARTITION BY battery_id ORDER BY y, m, d, hr, mi, sec) AS cycle_number
    FROM formatted_time
    WHERE type = 'impedance'
    """)

    display(spark.sql("""
        SELECT * FROM silver_impedance_metadata
        WHERE start_datetime IS NOT NULL
        LIMIT 50
    """))

In [0]:
if RUN_MEDALLION_ETL:
    spark.sql("""
    CREATE OR REPLACE TABLE gold_battery_pinn_database AS
    SELECT
      d.battery_id,
      d.type,
      d.filename,
      d.ambient_temperature,
      d.start_datetime,
      d.cycle_number,
      CAST(d.capacity AS FLOAT) AS capacity,
      CAST(i.Re AS FLOAT) AS Re,
      CAST(i.Rct AS FLOAT) AS Rct
    FROM silver_discharge_metadata d
    LEFT JOIN silver_impedance_metadata i
      ON d.battery_id = i.battery_id
      AND i.start_datetime <= d.start_datetime
    -- Keep only the first occurrence of each filename
    -- Sorting by start_datetime ensures we keep the most logical chronological record
    QUALIFY ROW_NUMBER() OVER(PARTITION BY d.filename ORDER BY d.start_datetime ASC) = 1
    """)

    display(spark.sql("""
        SELECT * FROM gold_battery_pinn_database
        WHERE start_datetime IS NOT NULL
        ORDER BY filename
        LIMIT 50
    """))
else:
    # Sanity check that the reused table actually exists when skipping ETL
    assert spark.catalog.tableExists("gold_battery_pinn_database"), (
        "gold_battery_pinn_database does not exist yet -- set RUN_MEDALLION_ETL=True "
        "and rerun this section once before skipping it on future runs."
    )
    print("gold_battery_pinn_database found -- reusing existing table.")

In [0]:
# Quick sanity check on the Gold table (safe to run whether or not ETL just ran)
display(spark.sql("""
    SELECT `battery_id`, COUNT(*) AS `count`
    FROM `workspace`.`default`.`gold_battery_pinn_database`
    WHERE `ambient_temperature` = 24
    GROUP BY `battery_id`
    ORDER BY `count` DESC
"""))

## 3. Feature Engineering (SoC / SoH / Cycle Downsampling)

**Fix applied:** the original silently skipped cycles below the `1.4 Ah` capacity floor and any
cycle shorter than 50 samples, with no visibility into how much data was dropped. This version
counts and prints every skip reason, plus the final retained fraction, so silent data loss can't
go unnoticed.

In [0]:
# --- Configurations ---
cutoff_map = {
    'B0005': 2.7,'B0006': 2.5,'B0007': 2.2,'B0018': 2.5,'B0025': 2.0, 'B0026': 2.2,
    'B0027': 2.5,'B0028': 2.7,'B0029': 2.0,'B0030': 2.2,'B0031': 2.5,'B0032': 2.7,
    'B0033': 2.0,'B0034': 2.2,'B0036': 2.7,'B0038': 2.2,'B0039': 2.5,'B0040': 2.7,
    'B0041': 2.0,'B0042': 2.2,'B0043': 2.5,'B0044': 2.7,'B0045': 2.0,'B0046': 2.2,
    'B0047': 2.5,'B0048': 2.7,'B0053': 2.0,'B0054': 2.2,'B0055': 2.5,'B0056': 2.7
}

target_batteries = ['B0005', 'B0006', 'B0007', 'B0018', 'B0025', 'B0026', 'B0027',
                    'B0028', 'B0029', 'B0030', 'B0031', 'B0032', 'B0033', 'B0034',
                    'B0036', 'B0038', 'B0039', 'B0040', 'B0041', 'B0042', 'B0043',
                    'B0044', 'B0045', 'B0046', 'B0047', 'B0048', 'B0053', 'B0054',
                    'B0055', 'B0056']

# Minimum capacity (Ah) for a cycle to be considered "healthy enough" to trust.
# Named as a constant rather than a bare literal so its purpose is documented
# and it's easy to tune per-chemistry.
MIN_VALID_CAPACITY_AH = 1.4
MIN_SAMPLES_PER_CYCLE = 50

# --- Query the Gold Database ---
query = f"""
    SELECT * FROM gold_battery_pinn_database
    WHERE battery_id IN ({','.join([f"'{b}'" for b in target_batteries])})
"""
discharge_metadata = spark.sql(query).toPandas()

# Run once outside the loop
max_caps_df = spark.sql("""
    SELECT battery_id, max(capacity) as max_cap
    FROM `workspace`.`default`.`gold_battery_pinn_database`
    GROUP BY battery_id
""").toPandas().set_index('battery_id')

processed_dfs = []

# --- Skip-reason counters (fix: visibility into data loss) ---
skip_counts = {
    'file_not_found': 0,
    'too_short': 0,
    'low_capacity': 0,
}

# --- Process Discharge Cycles ---
for _, row in tqdm(discharge_metadata.iterrows(), total=len(discharge_metadata)):
    file_path = f"/Volumes/workspace/default/pinn/cleaned_battery_dataset/data/{row['filename']}"
    if not os.path.exists(file_path):
        skip_counts['file_not_found'] += 1
        continue

    # Load raw time-series data
    df = pd.read_csv(file_path).copy()

    # Truncate at voltage limit
    v_limit = cutoff_map.get(row['battery_id'], 2.7)
    cutoff_idx = df[df['Voltage_measured'] < v_limit].index.min()
    truncated_df = df if pd.isna(cutoff_idx) else df.iloc[:cutoff_idx].copy()

    if len(truncated_df) < MIN_SAMPLES_PER_CYCLE:
        skip_counts['too_short'] += 1
        continue

    # Calculate Capacity & SoC
    db_capacity = float(row['capacity'])
    max_capacity = max_caps_df.loc[row['battery_id'], 'max_cap']

    # Only process cycles with valid capacity
    if db_capacity <= MIN_VALID_CAPACITY_AH:
        skip_counts['low_capacity'] += 1
        continue

    truncated_df['Time_diff_hr'] = truncated_df['Time'].diff().fillna(0) / 3600
    truncated_df['Delta_Q'] = abs(truncated_df['Current_measured'] * truncated_df['Time_diff_hr'])
    truncated_df['Cumulative_Q'] = truncated_df['Delta_Q'].cumsum()
    truncated_df['SoC'] = 100 * (1 - truncated_df['Cumulative_Q'] / db_capacity)

    # Downsample to 50 points
    num_bins = 50
    truncated_df['bin'] = pd.cut(truncated_df.index, bins=num_bins, labels=False)

    agg_cycle = truncated_df.groupby('bin').agg({
        'Time': 'mean',
        'Voltage_measured': 'mean',
        'Current_measured': 'mean',
        'Temperature_measured': 'mean',
        'SoC': 'mean'
    }).reset_index()

    # Add Database-derived Metadata
    agg_cycle['battery_id'] = row['battery_id']
    agg_cycle['cycle_number'] = row['cycle_number']
    agg_cycle['SoH'] = float((db_capacity / max_capacity) * 100)
    agg_cycle['Re'] = float(row['Re']) if pd.notna(row['Re']) else None
    agg_cycle['Rct'] = float(row['Rct']) if pd.notna(row['Rct']) else None
    # Fix: carry ambient_temperature through -- previously dropped here, which
    # meant every cycle was implicitly assumed to share one global thermal
    # reference despite the NASA dataset spanning multiple ambient test groups
    # (~4C / 24C / 43C). See Section 4 for how this is used in the PDE.
    agg_cycle['ambient_temperature'] = float(row['ambient_temperature']) if pd.notna(row['ambient_temperature']) else None
    agg_cycle['Time_normalized'] = agg_cycle['Time'] / agg_cycle['Time'].max()

    processed_dfs.append(agg_cycle)

# --- Final Dataset Creation ---
total_rows = len(discharge_metadata)
total_kept = len(processed_dfs)
total_skipped = sum(skip_counts.values())

print("--- Cycle processing summary ---")
print(f"Total candidate cycles:   {total_rows}")
print(f"Kept:                     {total_kept} ({100 * total_kept / max(total_rows, 1):.1f}%)")
print(f"Skipped (file not found): {skip_counts['file_not_found']}")
print(f"Skipped (too short):      {skip_counts['too_short']}")
print(f"Skipped (low capacity):   {skip_counts['low_capacity']}")

if total_skipped > 0.5 * total_rows:
    print("WARNING: over half of all cycles were skipped — check cutoff_map / MIN_VALID_CAPACITY_AH.")

if processed_dfs:
    full_dataset = pd.concat(processed_dfs, ignore_index=True)

    # Save the processed data back into a NEW Gold Table for the PINN
    spark_processed = spark.createDataFrame(full_dataset)
    # Fix: overwriteSchema is needed (not mergeSchema) when the table already
    # exists with an older schema (e.g. missing ambient_temperature from a
    # prior run) and Table ACLs block automatic schema migration.
    spark_processed.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("final_pinn_training_data")

    print("Training database 'final_pinn_training_data' created with target batteries.")
else:
    raise RuntimeError("No cycles survived processing — check the skip-reason counts above.")

In [0]:
spark.sql("""
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cycle_number ORDER BY SoH DESC) as rn
        FROM final_pinn_training_data
    )
    WHERE rn = 1
    ORDER BY SoH DESC
    LIMIT 10
""").show()

**Note:** the query above is a `ROW_NUMBER() ... ORDER BY SoH DESC` preview, so it only ever
surfaces the highest-SoH row per `cycle_number` -- it will always look like SoH is clustered near
100%, regardless of the actual dataset distribution. The cell below queries the real, unbiased
distribution instead.

In [0]:
# --- Unbiased distribution check (SoH + ambient_temperature) ---
spark.sql("""
    SELECT
        ambient_temperature,
        COUNT(*) AS n_cycles,
        ROUND(MIN(SoH), 1) AS min_SoH,
        ROUND(AVG(SoH), 1) AS avg_SoH,
        ROUND(MAX(SoH), 1) AS max_SoH
    FROM final_pinn_training_data
    GROUP BY ambient_temperature
    ORDER BY ambient_temperature
""").show()

spark.sql("""
    SELECT
        FLOOR(SoH / 10) * 10 AS SoH_bucket,
        COUNT(*) AS n_cycles
    FROM final_pinn_training_data
    GROUP BY FLOOR(SoH / 10) * 10
    ORDER BY SoH_bucket
""").show()

## 4. Physics Definition

**Fixes applied in this section:**
- The linear OCV approximation `U_ocv = 2.7 + 0.015 * SoC` has been replaced with a 4th-order
  polynomial fit against the pipeline's own `(SoC, Voltage_measured)` low-rate discharge data.
- **Ambient temperature is now a model input.** Instead of one global `T_ref = 25°C` baseline for
  every battery, each point's thermal reference is `T_ref_point = ambient_temperature + 273.15`,
  taken from a new 4th spatial-parameter input dimension (`[space, SoH, Re, Ambient, time]`).
  `T_delta` (the network's output) is now interpreted as deviation from *that battery's own*
  ambient condition rather than a single fixed 25°C point. This matters because the dataset spans
  multiple distinct ambient test groups (~4°C / 24°C / 43°C) — forcing them all through one global
  reference is a plausible reason the PDE residual losses were diverging during training.

If you have access to true open-circuit (near-zero-current, e.g. C/20) reference discharges, refit
`ocv_coeffs` against those instead of the full-rate `Voltage_measured` column for a cleaner OCV
curve — the polynomial fit here is intentionally kept flexible so you can swap the fitting data
without touching the PDE code.

### Arrhenius numerics — diagnosing why `A_sei` wasn't learning

**Fix applied:** in the previous run, `A_sei_value` stayed completely flat at its untrained default
(1×10¹⁰) for all 15,000 steps. The root cause isn't the loss weighting — it's that the reaction
rate `rate = A_sei * c * exp(-Ea_sei / (R*T))` starts out **astronomically small** at realistic
battery temperatures. With `Ea_sei = 1.35e5` J/mol, the exponent is around −51 to −54 across the
0–45°C operating range — deep past the old `clamp(min=-50)` guard, and `exp(-50) ≈ 1.9e-22`. That
makes `rate ≈ A_sei(1e10) × 1.9e-22 ≈ 1.9e-12` — a term so many orders of magnitude smaller than the
Joule-heating term (`O(1)`) it's added to that its contribution to the loss gradient is negligible
from the very first step, regardless of loss weighting.

The cell below prints the actual exponent/rate magnitudes across the dataset's realistic
temperature range (including the 150°C abuse-test extreme) so this is diagnosed with real numbers
rather than assumed, then picks a better-informed starting value for `log_A_sei` so the reaction
rate starts at a magnitude the network can actually learn from.

In [0]:
# ---------------------------------------------------------
#  Diagnostic: is the Arrhenius rate term numerically observable?
# ---------------------------------------------------------
R_gas = 8.314        # Gas constant (J/mol*K)
Ea_sei = 1.35e5      # Activation energy (J/mol)

test_temps_C = np.array([0, 4, 24, 35, 43, 60, 80, 100, 150])
test_temps_K = test_temps_C + 273.15
exponents = -Ea_sei / (R_gas * test_temps_K)

print("Temp (C) | exponent (unclamped) | exp(exponent)")
for t_c, exp_val in zip(test_temps_C, exponents):
    print(f"{t_c:8.1f} | {exp_val:20.2f} | {np.exp(exp_val):.3e}")

print(f"\nExponent range across 0-150C: [{exponents.min():.1f}, {exponents.max():.1f}]")
print("The original clamp bound was min=-50 -- this SATURATES for most of the")
print("0-45C operating range, cutting the exponential to a fixed floor and")
print("compressing what should be meaningful temperature-dependent variation.")

# Recommend a starting A_sei such that the reaction rate is O(1) -- comparable
# to the Joule-heating term -- at a representative mid-range operating
# temperature, so gradients w.r.t. log_A_sei are non-negligible from step 0.
T_reference_for_init = 308.15  # ~35C, representative of the dataset's operating range
exponent_at_ref = -Ea_sei / (R_gas * T_reference_for_init)
recommended_log_A_sei = -exponent_at_ref / np.log(10)  # solves A_sei * exp(exponent) = 1

print(f"\nRecommended log10(A_sei) initial value (targets rate~O(1) at "
      f"{T_reference_for_init - 273.15:.0f}C): {recommended_log_A_sei:.2f}")
print(f"(Previous hardcoded init was 10.0 -- {abs(recommended_log_A_sei - 10.0):.1f} orders of "
      f"magnitude off, which explains why gradient signal was effectively zero.)")

In [0]:
# ---------------------------------------------------------
#  Physics Constants & Parameters
# ---------------------------------------------------------
H_sei = 2.57e5       # Enthalpy of reaction (J/kg)
rho = 2000.0         # Density (kg/m3)
Cp = 1000.0          # Heat capacity (J/kg*K)
k_cond = 1.0         # Thermal conductivity (W/m*K)
T_ref = 298.15       # Baseline Kelvin (25 degC) -- kept for reference/backward compat only;
                      # the PDE now uses a per-point ambient reference (see Section 4's
                      # sei_thermal_pde, which reads T_ref_point from the Ambient input column
                      # instead of this global constant).

# Fix: widened from (-50, 50) to (-70, 50) so the exponential guard no longer
# saturates across the dataset's realistic 0-150C range (see diagnostic
# above) -- it's now purely a numerical guard against exp() overflow, not
# something that silently clips gradient signal during normal training.
EXPONENT_CLAMP_MIN = -70
EXPONENT_CLAMP_MAX = 50

# Fix: A_sei is initialized using the diagnostic above instead of an
# arbitrary guess, so the reaction rate starts at a magnitude the network can
# actually learn from, rather than starting ~12+ orders of magnitude too small.
log_A_sei = dde.Variable(float(recommended_log_A_sei))

# NOTE: even with this fix, keep an eye on whether A_sei moves meaningfully
# during training. SEI-driven heat is only physically significant at abuse
# temperatures, not normal operating temperatures -- so it's possible this
# parameter is only weakly identifiable from single-cycle discharge data in
# the first place. If A_sei still barely moves after this fix, that's a sign
# to treat it as a fixed literature value rather than a calibrated one.

In [0]:
# ---------------------------------------------------------
#  Data Preparation & Database Querying
# ---------------------------------------------------------
# Load data from the processed SQL database
query = "SELECT * FROM final_pinn_training_data"
df = spark.sql(query).toPandas()

#  Replace any rogue 'Infinite' values with NaN
df = df.replace([np.inf, -np.inf], np.nan)
#  Drop any row that contains missing data in our critical columns
df = df.dropna(subset=[
    'Time', 'Time_normalized', 'Current_measured',
    'Voltage_measured', 'SoC', 'SoH', 'Re', 'Temperature_measured',
    'ambient_temperature'
])

# Constants for scaling
t_max_seconds = df['Time'].max()

# Strictly monotonic interpolation:
# Since we are training a global model across many batteries,
# we interpolate the average characteristic profiles for Current, Voltage, and SoC.
# This provides the "Representative Discharge" the PINN learns from.

# Sort by normalized time to ensure 1D interpolation works correctly
df_sorted = df.sort_values('Time_normalized')

# Group by normalized time to handle duplicates at specific time steps
df_agg = df_sorted.groupby('Time_normalized').agg({
    'Current_measured': 'mean',
    'Voltage_measured': 'mean',
    'SoC': 'mean'
}).reset_index()

# NOTE: the scipy interp1d objects previously built here have been removed.
# See the "GPU-native interpolation" cell below, which replaces them with
# torch tensors + torch.searchsorted so PDE residual evaluation never has to
# round-trip through the CPU during training.

In [0]:
# ---------------------------------------------------------
#  OCV(SoC) fit  (fix: replaces the linear U_ocv = 2.7 + 0.015*SoC)
# ---------------------------------------------------------
# Fit a 4th-order polynomial to (SoC, Voltage) pairs from the aggregated
# representative discharge curve. This is a pragmatic stand-in for a true
# low-rate OCV curve -- if reference C/20 discharges are available, refit
# against those instead for a cleaner, rate-independent OCV estimate.
_soc_fit = df_agg['SoC'].values / 100.0          # normalize to [0, 1]
_v_fit = df_agg['Voltage_measured'].values

_soc_fit_clipped = np.clip(_soc_fit, 1e-3, 1 - 1e-3)
ocv_coeffs = np.polyfit(_soc_fit_clipped, _v_fit, deg=4)

print(f"Fitted OCV polynomial coefficients (deg=4, highest power first): {ocv_coeffs}")

def ocv_from_soc_torch(soc_tensor):
    """Evaluate the fitted OCV(SoC) polynomial on a torch tensor."""
    soc = torch.clamp(soc_tensor, 1e-3, 1 - 1e-3)
    ocv = torch.zeros_like(soc)
    degree = len(ocv_coeffs) - 1
    for power, coeff in zip(range(degree, -1, -1), ocv_coeffs):
        ocv = ocv + float(coeff) * torch.pow(soc, power)
    return ocv

### GPU-native interpolation (fix: replaces per-call `scipy.interp1d`)

The original `get_real_time_physics` called `.detach().cpu().numpy()` on every invocation, ran
`scipy.interp1d` on the CPU, then moved the result back to the training device. Since this function
is called on every PDE residual evaluation, that CPU/GPU sync happened thousands of times during
training and was very likely the actual training-speed bottleneck — more so than GPU tier choice.

The replacement below precomputes the lookup tables as tensors **once**, on the training device, and
does the interpolation with `torch.searchsorted` so the whole call stays on-device.

In [0]:
# ---------------------------------------------------------
#  GPU-native interpolation tables (fix: replaces scipy interp1d)
# ---------------------------------------------------------
# Precompute sorted breakpoints + values as tensors, on the training device,
# once -- instead of re-running scipy interp1d + a CPU round-trip on every
# single PDE residual call during training.

_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

t_pts_t   = torch.tensor(df_agg['Time_normalized'].values, dtype=torch.float32, device=_device)
I_pts_t   = torch.tensor(df_agg['Current_measured'].values, dtype=torch.float32, device=_device)
V_pts_t   = torch.tensor(df_agg['Voltage_measured'].values, dtype=torch.float32, device=_device)
SoC_pts_t = torch.tensor(df_agg['SoC'].values, dtype=torch.float32, device=_device)

def torch_interp(x, xp, fp):
    """
    1D linear interpolation, fully on-device (mirrors np.interp / scipy interp1d
    with linear extrapolation at the edges). x, xp, fp are all torch tensors.
    xp must be sorted ascending (guaranteed since df_agg is sorted by Time_normalized).
    """
    x = x.flatten()
    # Find insertion indices, clamp to valid interior range for interpolation
    idx = torch.searchsorted(xp, x, right=True)
    idx = torch.clamp(idx, 1, len(xp) - 1)

    x0 = xp[idx - 1]
    x1 = xp[idx]
    y0 = fp[idx - 1]
    y1 = fp[idx]

    # Linear interpolation; denom guarded against duplicate breakpoints
    denom = (x1 - x0)
    denom = torch.where(denom.abs() < 1e-12, torch.ones_like(denom), denom)
    slope = (y1 - y0) / denom

    out = y0 + slope * (x - x0)

    # Linear extrapolation beyond the table edges (matches fill_value="extrapolate")
    below = x < xp[0]
    above = x > xp[-1]
    if below.any():
        edge_slope = (fp[1] - fp[0]) / (xp[1] - xp[0] + 1e-12)
        out = torch.where(below, fp[0] + edge_slope * (x - xp[0]), out)
    if above.any():
        edge_slope = (fp[-1] - fp[-2]) / (xp[-1] - xp[-2] + 1e-12)
        out = torch.where(above, fp[-1] + edge_slope * (x - xp[-1]), out)

    return out.view(-1, 1)


def get_real_time_physics(t_norm):
    """
    Mapping function for the PINN's physics loss.
    Converts normalized time [0,1] into the specific I, V, and OCV
    required to solve the heat equation during training.

    Fully GPU-native: no .detach()/.cpu()/.numpy() round-trip, so this no
    longer forces a CPU<->GPU sync on every PDE residual evaluation.
    """
    t_flat = t_norm.view(-1).to(_device)

    I_val = torch_interp(t_flat, t_pts_t, I_pts_t).to(t_norm.device)
    V_val = torch_interp(t_flat, t_pts_t, V_pts_t).to(t_norm.device)
    SoC_val = torch_interp(t_flat, t_pts_t, SoC_pts_t).to(t_norm.device)

    # OCV: fitted nonlinear curve (from the fix above), evaluated in torch
    U_ocv = ocv_from_soc_torch(SoC_val / 100.0)

    return I_val, V_val, U_ocv

**Optional sanity check** — before relying on this for a full training run, verify the two
interpolation methods agree within floating-point tolerance on the same input points:

```python
from scipy.interpolate import interp1d

_ref_interp_I = interp1d(df_agg['Time_normalized'].values, df_agg['Current_measured'].values, fill_value="extrapolate")
_test_t = np.linspace(0, 1, 200).astype(np.float32)

_ref = _ref_interp_I(_test_t)
_new = torch_interp(torch.tensor(_test_t, device=_device), t_pts_t, I_pts_t).cpu().numpy().flatten()

print("Max abs difference (interp_I vs torch_interp):", np.max(np.abs(_ref - _new)))
```


### 4D + Time Coupled PDE System

Same physics as before, but `T_ref` is no longer a single global constant — each point now carries
its own ambient reference via a new input dimension. Input column order is
`[space(0), SoH(1), Re(2), Ambient(3), time(4)]`, so the time-derivative jacobians move from
`j=3` to `j=4` (Ambient is a static parameter dimension like SoH/Re, not something with a
meaningful time-derivative).

In [0]:
# ---------------------------------------------------------
# 4D + Time PDE System
# ---------------------------------------------------------

def sei_thermal_pde(x, y):
    T_delta = y[:, 0:1]
    c = y[:, 1:2]

    # 1. Per-point ambient reference (fix: replaces the single global T_ref).
    # x[:, 3:4] is the Ambient input dimension, in Celsius; convert to Kelvin.
    T_ref_point = x[:, 3:4] + 273.15

    # 2. Scale and Guard Temperature relative to that point's own ambient.
    T = T_delta * 50.0 + T_ref_point
    T_safe = torch.clamp(T, min=273.15, max=600.0)

    # 3. Guard Concentration (Health cannot be negative or above 100%)
    c_safe = torch.clamp(c, min=0.0, max=1.0)

    # 4. Gradients -- time is now the 5th input column (index 4)
    dT_t = dde.grad.jacobian(y, x, i=0, j=4) / t_max_seconds
    dc_t = dde.grad.jacobian(y, x, i=1, j=4) / t_max_seconds
    dT_xx = dde.grad.hessian(y, x, i=0, j=0)

    # 5. Physics Inputs -- time is x[:, 4:5] now
    I, V, U_ocv = get_real_time_physics(x[:, 4:5])
    actual_A_sei = torch.pow(10.0, log_A_sei)
    joule_heat = torch.abs(I * (V - U_ocv))

    # 6. Arrhenius Rate with Exponential Guard
    exponent = -Ea_sei / (R_gas * T_safe + 1e-6)
    rate = actual_A_sei * c_safe * torch.exp(torch.clamp(exponent, min=EXPONENT_CLAMP_MIN, max=EXPONENT_CLAMP_MAX))

    # 7. Thermal Balance (Using a small epsilon 1e-6 to prevent division by zero)
    thermal_mass = (rho * Cp) + 1e-6
    res_thermal = (dT_t * 50.0) - (k_cond * dT_xx * 50.0 + joule_heat + H_sei * rate) / thermal_mass

    res_sei = dc_t + rate

    return [res_thermal, res_sei]

## 5. Geometry & Boundary/Initial Conditions

**Fix applied:** the domain is now a 4D hypercube (`space, SoH, Re, Ambient`) + time, instead of
3D + time. The `Ambient` dimension spans roughly the dataset's ambient test groups
(0–45°C, covering the NASA ~4°C / 24°C / 43°C groups with margin). The initial-condition check and
observation points are updated to match the new column order
`[space(0), SoH(1), Re(2), Ambient(3), time(4)]`.

In [0]:
# ---------------------------------------------------------
# Geometry & Robust Initial Conditions
# ---------------------------------------------------------

# 1. Define Geometry -- 4D hypercube (space, SoH, Re, Ambient) + time
param_geom = dde.geometry.Hypercube(xmin=[0, 0.5, 0.0, 0.0], xmax=[1, 1.2, 12.0, 45.0])
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(param_geom, timedomain)

# 2. Fuzzy ICs -- time is now column index 4
def on_initial_condition(x, on_initial):
    # Using on_initial (the built-in flag) OR a small temporal window
    return on_initial or (x[4] <= 0.01)

ic_T = dde.icbc.IC(geomtime, lambda x: 0, on_initial_condition, component=0)
ic_c = dde.icbc.IC(geomtime, lambda x: 1.0, on_initial_condition, component=1)

# --- Observation points for the data-fitting boundary condition ---
obs_points = np.column_stack((
    np.ones(len(df)),                     # Space (x=1)
    df['SoH'].values / 100.0,             # SoH
    df['Re'].values * 10.0,               # Re
    df['ambient_temperature'].values,     # Ambient (Celsius) -- fix: new input
    df['Time_normalized'].values          # Time (t)
))

# obs_values: T_delta target = (measured T - THAT ROW'S ambient reference) / 50
#
# v6 fix (unit bug): NASA's Temperature_measured column is in Celsius, same
# as ambient_temperature -- confirmed by the Section 6 diagnostic, which
# showed bin-0 (Temperature_measured - ambient) averaging around -272 K,
# i.e. off by almost exactly 273.15. The previous version added +273.15 to
# ambient_temperature only, implicitly treating Temperature_measured as
# already being in Kelvin when it isn't. Since this is a *difference* of two
# Celsius values, no Kelvin conversion is needed at all here -- the +273.15
# would cancel out algebraically even if both sides were correctly converted
# ((T_C + 273.15) - (ambient_C + 273.15) = T_C - ambient_C). The PDE's own
# Arrhenius calculation (sei_thermal_pde) already converts to Kelvin
# correctly on its own side, so no change is needed there -- only this
# formula, which feeds the training target T_delta is compared against.
obs_values = (
    (df['Temperature_measured'].values - df['ambient_temperature'].values) / 50.0
).reshape(-1, 1)

# Synchronize Data Types
obs_points = obs_points.astype(np.float32)
obs_values = obs_values.astype(np.float32)

observe_T = dde.icbc.PointSetBC(obs_points, obs_values, component=0)

## 6. Model Architecture & Training

**Fixes applied in this section:**
- Network input dimension is now **5** (`space, SoH, Re, Ambient, time`), matching the new geometry.
- `log_A_sei` is saved *inside* the same checkpoint dict as the network weights, instead of only the
  weights being persisted, so it can be correctly restored in Section 7.
- **Loss weights rebalanced.** The original `[1, 1, 10, 10, 100]` gave the data-fit term
  10–100x the weight of either PDE residual — consistent with the MLflow run showing
  `loss_comp_0`/`loss_comp_1` (the PDE residuals) *increasing* over training while
  `log_A_sei` never moved from its untrained default. Weights are rebalanced here to
  `[10, 10, 5, 5, 20]` so the physics terms get real gradient signal; watch whether
  `loss_comp_0`/`loss_comp_1` start decreasing and whether `A_sei_value` starts moving before
  trusting the calibrated value.
- **Metric callback fixed.** The original `mae_temp_metric`/`l2_temp_error` relied on
  `train_state.metrics_test`, which mostly samples unlabeled PDE-domain points — `y_true` was
  `None` almost every call, so the metric silently returned `0.0` for the entire run (visible as
  the flat `MAE_Kelvin_Scaled`/`L2_Relative_Error` lines in the MLflow plot). The callback below
  computes MAE/L2 directly against the real `obs_points`/`obs_values` observations instead.

In [0]:
import mlflow
import torch
import deepxde as dde

# ---------------------------------------------------------
# Model Architecture & Training
# ---------------------------------------------------------
# Metrics for Battery Digital Twin
def mae_temp_metric(y_true, y_pred):
    # If DeepXDE is testing on PDE points (no NASA labels), y_true is None
    if y_true is None:
        return torch.tensor(0.0)

    # y_pred has 2 columns: [T_delta, c]
    # y_true has 1 column: [Temperature_measured]
    # We slice y_pred to only compare the Temperature component
    return torch.mean(torch.abs(y_true - y_pred[:, 0:1]))

def l2_temp_error(y_true, y_pred):
    if y_true is None:
        return torch.tensor(0.0)

    # Compare only the temperature components
    return torch.norm(y_true - y_pred[:, 0:1]) / (torch.norm(y_true) + 1e-7)

# Data Object Configuration
# num_test=2000 provides the 'y_true' values needed for the metrics to function
data = dde.data.TimePDE(
    geomtime,
    sei_thermal_pde,
    [ic_T, ic_c, observe_T],
    num_domain=5000,
    num_boundary=1000,    # Doubled this to ensure t=0 is captured
    anchors=obs_points,
    num_test=2000
)

# Network Architecture
# 5 Inputs: [Space, SoH, Re, Ambient, Time] -> 4 Hidden Layers -> 2 Outputs: [T_delta, c]
# Use the float32 casting for the network weights
net = dde.nn.FNN([5] + [64] * 4 + [2], "tanh", "Glorot normal")
net.float()  # Force weights to float32
model = dde.model.Model(data, net)

# Rebalanced loss weights (fix: physics terms previously dwarfed by the data term)
# Order matches [ic_T, ic_c, observe_T] + the 2 PDE residuals returned by
# sei_thermal_pde -- DeepXDE orders PDE-residual weights first, then the
# icbc list, so this is [res_thermal, res_sei, ic_T, ic_c, observe_T].
#
# v5 fix: ic_T/ic_c raised from 5 to 30. The previous run's MLflow curves
# showed loss_comp_2 (ic_T) plateau flat at ~0.849 for the second half of
# training and through L-BFGS, instead of continuing to decrease -- a sign
# the data-fit term (observe_T, weight 20) was winning a conflict against the
# t=0 initial condition rather than both being satisfied. Raising the IC
# weight relative to observe_T gives the optimizer more reason to actually
# resolve that conflict instead of settling on a compromise. See the
# diagnostic cell after training, which checks whether bin-0 (the first
# downsampled point per cycle) already shows measurable warm-up relative to
# ambient -- if so, that's a data-quality reason ic_T may never reach exactly
# zero, independent of loss weighting.
TRAIN_LOSS_WEIGHTS = [10, 10, 30, 30, 20]

# Custom MLflow Logging Callback
class MLflowLoggingCallback(dde.callbacks.Callback):
    def on_epoch_end(self):
        # Log every 1000 steps to avoid cluttering Databricks logs
        if self.model.train_state.step % 1000 == 0:
            # Log the learned physical parameter A_sei
            current_A = 10 ** log_A_sei.detach().item()
            mlflow.log_metric("A_sei_value", current_A, step=self.model.train_state.step)

            # Log all 5 loss components (PDE 1, PDE 2, IC 1, IC 2, Data)
            for i, loss in enumerate(self.model.train_state.loss_train):
                mlflow.log_metric(f"loss_comp_{i}", float(loss), step=self.model.train_state.step)

            # Fix: compute MAE/L2 directly against the real NASA temperature
            # observations instead of relying on train_state.metrics_test,
            # which mostly evaluates on unlabeled PDE-domain points (y_true=None)
            # and was silently returning 0.0 for the whole run.
            pred = self.model.predict(obs_points)
            mae = np.mean(np.abs(obs_values[:, 0] - pred[:, 0]))
            l2 = np.linalg.norm(obs_values[:, 0] - pred[:, 0]) / (np.linalg.norm(obs_values[:, 0]) + 1e-7)
            mlflow.log_metric("MAE_Kelvin_Scaled", float(mae), step=self.model.train_state.step)
            mlflow.log_metric("L2_Relative_Error", float(l2), step=self.model.train_state.step)


# Execution within MLflow Context
with mlflow.start_run(run_name="Global_Battery_Digital_Twin"):

    print("Starting Adam Optimization...")
    # Stage 1: Adam Optimizer
    model.compile(
        "adam",
        lr=1e-4,  # Lower learning rate to prevent early 'jump' to NaN
        loss_weights=TRAIN_LOSS_WEIGHTS,
        external_trainable_variables=log_A_sei,
        metrics=[mae_temp_metric, l2_temp_error]
    )

    # Train with Adam for 15,000 iterations to stabilize the weights
    losshistory_adam, train_state_adam = model.train(iterations=15000, callbacks=[MLflowLoggingCallback()])

    # Stage 2: L-BFGS Fine-Tuning
    print("Starting L-BFGS Refinement...")
    # Keep the same rebalanced weights for the refinement stage rather than
    # reverting to defaults, so the physics terms stay meaningfully weighted.
    model.compile("L-BFGS", loss_weights=TRAIN_LOSS_WEIGHTS, external_trainable_variables=log_A_sei)

    # Increase the tolerance so L-BFGS doesn't give up early
    dde.optimizers.config.set_LBFGS_options(maxiter=2000, ftol=1e-9)

    # Train with L-BFGS to fine-tune the stiff PDE gradients
    losshistory_lbfgs, train_state_lbfgs = model.train()

    # Final Artifact Logging
    final_A = 10 ** log_A_sei.detach().item()
    mlflow.log_param("Final_Calibrated_A_sei", f"{final_A:.2e}")

    # --- Fix: persist log_A_sei alongside the network weights ---
    # Previously only net.state_dict() was saved, so restoring the model for
    # inference reset log_A_sei to its untrained default (10.0) rather than
    # the value actually calibrated during training.
    torch.save({
        "model_state_dict": net.state_dict(),
        "log_A_sei": log_A_sei.detach().item(),
        "ocv_coeffs": ocv_coeffs.tolist(),
    }, "battery_pinn_model.pt")
    mlflow.log_artifact("battery_pinn_model.pt")

print(f"Training successfully completed. Frequency factor calibrated to: {final_A:.2e}")
model.save("global_pinn_model")

### Diagnostic — is the `ic_T` plateau caused by warm-up already present in bin-0?

Checks whether the first downsampled point per cycle (`bin=0`) already shows a temperature
noticeably above ambient (i.e. the discharge curve was already warming up by the time it was
averaged into that first bin), which would put the `t=0` initial condition (`T_delta=0`) in
genuine tension with the data-fit term at the same nominal time point — independent of loss
weighting. Also compares the trained model's actual prediction at those bin-0 points against the
IC target of 0.

In [0]:
# ---------------------------------------------------------
#  Diagnostic: bin-0 warm-up vs the ic_T=0 target
# ---------------------------------------------------------
bin0_mask = (
    df['Time_normalized'].values
    == df.groupby(['battery_id', 'cycle_number'])['Time_normalized'].transform('min').values
)
bin0_df = df[bin0_mask]

# v6 fix (unit bug): both Temperature_measured and ambient_temperature are in
# Celsius -- no +273.15 conversion needed for a plain difference (see the
# obs_values fix in Section 5 for the full explanation). The previous version
# of this diagnostic mixed units, producing a nonsensical ~-272K result.
delta_at_bin0 = bin0_df['Temperature_measured'].values - bin0_df['ambient_temperature'].values
print(f"bin-0 Time_normalized range: [{bin0_df['Time_normalized'].min():.5f}, {bin0_df['Time_normalized'].max():.5f}]")
print(f"bin-0 (Temperature_measured - ambient), degrees C -- if far from 0, bin-0 already shows warm-up:")
print(f"  mean={delta_at_bin0.mean():.2f}C, std={delta_at_bin0.std():.2f}C, "
      f"min={delta_at_bin0.min():.2f}C, max={delta_at_bin0.max():.2f}C")

# Compare the trained model's prediction at those same bin-0 points against
# the ic_T target of 0 (in the network's *50-scaled output units).
bin0_obs_points = obs_points[bin0_mask]
bin0_pred = model.predict(bin0_obs_points)
print(f"\nModel's predicted T_delta at bin-0 points (target is 0 under ic_T):")
print(f"  mean={bin0_pred[:, 0].mean():.3f}, std={bin0_pred[:, 0].std():.3f}  "
      f"(scaled units; multiply by 50 for degrees C)")
print(f"  In degrees C: mean={bin0_pred[:, 0].mean() * 50:.2f}C")

print("""
Interpretation:
- If 'bin-0 (Temp - ambient)' is already several degrees above zero, the observe_T data
  target at that point genuinely disagrees with ic_T's T_delta=0 requirement -- some
  residual ic_T loss is then expected/unavoidable at that data resolution, and a lower
  (but nonzero) plateau may be the correct converged answer, not a training failure.
- If the model's bin-0 prediction is well above what the data itself shows at bin-0,
  that instead points to the loss weighting/optimization -- worth trying an even higher
  ic_T/ic_c weight, or a physics-warm-up phase (train PDE+IC only before introducing
  observe_T).
""")

## 7. Digital Twin Restoration & Inference

**Fixes applied:**
- **Self-contained.** This section now redefines every physics constant, interpolation table, and
  helper function it needs (constants, `torch_interp`/`get_real_time_physics`, `ocv_from_soc_torch`,
  `sei_thermal_pde`, the metric functions) by re-reading the persisted `final_pinn_training_data`
  table — it no longer silently depends on Sections 4–6 having just run in the same kernel session.
  You can restart the kernel and run **Section 1 (imports) → this section** directly, as long as
  the one-off Section 2/3 tables already exist and you have a trained run's `run_id`.
- **`run_id` is a placeholder you must fill in** — the cell below fails fast with a clear error if
  it's left unset, instead of silently trying to download a nonexistent MLflow run.
- The real `sei_thermal_pde` (rather than a `dummy_pde` placeholder) is reused to reconstruct the
  network architecture, so the PDE machinery is available if you want to check residuals later.
- `log_A_sei` and the fitted `ocv_coeffs` are restored from the checkpoint, not left at their
  untrained defaults — and if the checkpoint is missing either (e.g. an older run), this now raises
  a clear error rather than silently proceeding with wrong physics.
- Network input dimension updated to **5** to match the `Ambient` input added in Sections 5/6.
- **Bonus fix — the oven abuse test now actually uses its own temperature sweep.** The original
  cell computed `oven_t_c`/`oven_t_norm` (a 25→150°C ramp) but never fed it into `abuse_test` —
  predictions were made along the ordinary `[0,1]` time domain only, so the sweep was dead code.
  Now that `Ambient` is a real input dimension, the sweep drives the simulation directly: for a
  fixed exposure time, ambient stress is ramped from 25°C to 150°C and the model predicts core
  temperature and SEI stability at each stress level.
- An explicit note clarifies what `model.predict()` does and does not do at inference time.

In [0]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import mlflow
from mlflow.tracking import MlflowClient

# 1. SET BACKEND BEFORE IMPORTING DEEPXDE
os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde

# ---------------------------------------------------------
#  Config -- fill this in before running
# ---------------------------------------------------------
# Paste the MLflow run ID of the training run you want to restore (find it in
# the MLflow UI under Experiments, or in the Section 6 run output/URL).
run_id = "8868b518388b4708aafd2bcd3e014e9a"
artifact_path = "battery_pinn_model.pt"

if run_id == "PASTE_YOUR_MLFLOW_RUN_ID_HERE":
    raise ValueError(
        "Set run_id above to the MLflow run ID of your trained model before "
        "running this section (Experiments UI -> your run -> Run ID)."
    )

In [0]:
# ---------------------------------------------------------
#  Self-contained setup -- rebuilds everything this section needs from the
#  persisted final_pinn_training_data table, without requiring Sections 4-6
#  to have just run in this session. Safe after a kernel restart, as long as
#  the one-off Section 2/3 tables already exist.
# ---------------------------------------------------------
R_gas = 8.314
Ea_sei = 1.35e5
H_sei = 2.57e5
rho = 2000.0
Cp = 1000.0
k_cond = 1.0
EXPONENT_CLAMP_MIN = -70   # must match Section 4's diagnostic-informed value
EXPONENT_CLAMP_MAX = 50

query = "SELECT * FROM final_pinn_training_data"
df = spark.sql(query).toPandas()
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[
    'Time', 'Time_normalized', 'Current_measured',
    'Voltage_measured', 'SoC', 'SoH', 'Re', 'Temperature_measured',
    'ambient_temperature'
])
t_max_seconds = df['Time'].max()

df_sorted = df.sort_values('Time_normalized')
df_agg = df_sorted.groupby('Time_normalized').agg({
    'Current_measured': 'mean',
    'Voltage_measured': 'mean',
    'SoC': 'mean'
}).reset_index()

_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t_pts_t   = torch.tensor(df_agg['Time_normalized'].values, dtype=torch.float32, device=_device)
I_pts_t   = torch.tensor(df_agg['Current_measured'].values, dtype=torch.float32, device=_device)
V_pts_t   = torch.tensor(df_agg['Voltage_measured'].values, dtype=torch.float32, device=_device)
SoC_pts_t = torch.tensor(df_agg['SoC'].values, dtype=torch.float32, device=_device)

def torch_interp(x, xp, fp):
    x = x.flatten()
    idx = torch.searchsorted(xp, x, right=True)
    idx = torch.clamp(idx, 1, len(xp) - 1)
    x0 = xp[idx - 1]; x1 = xp[idx]
    y0 = fp[idx - 1]; y1 = fp[idx]
    denom = (x1 - x0)
    denom = torch.where(denom.abs() < 1e-12, torch.ones_like(denom), denom)
    slope = (y1 - y0) / denom
    out = y0 + slope * (x - x0)
    below = x < xp[0]
    above = x > xp[-1]
    if below.any():
        edge_slope = (fp[1] - fp[0]) / (xp[1] - xp[0] + 1e-12)
        out = torch.where(below, fp[0] + edge_slope * (x - xp[0]), out)
    if above.any():
        edge_slope = (fp[-1] - fp[-2]) / (xp[-1] - xp[-2] + 1e-12)
        out = torch.where(above, fp[-1] + edge_slope * (x - xp[-1]), out)
    return out.view(-1, 1)

def ocv_from_soc_torch(soc_tensor):
    # Note: ocv_coeffs is populated below once the checkpoint is restored --
    # this function isn't actually called until model.predict() runs, by
    # which point ocv_coeffs exists (Python looks up globals at call time).
    soc = torch.clamp(soc_tensor, 1e-3, 1 - 1e-3)
    ocv = torch.zeros_like(soc)
    degree = len(ocv_coeffs) - 1
    for power, coeff in zip(range(degree, -1, -1), ocv_coeffs):
        ocv = ocv + float(coeff) * torch.pow(soc, power)
    return ocv

def get_real_time_physics(t_norm):
    t_flat = t_norm.view(-1).to(_device)
    I_val = torch_interp(t_flat, t_pts_t, I_pts_t).to(t_norm.device)
    V_val = torch_interp(t_flat, t_pts_t, V_pts_t).to(t_norm.device)
    SoC_val = torch_interp(t_flat, t_pts_t, SoC_pts_t).to(t_norm.device)
    U_ocv = ocv_from_soc_torch(SoC_val / 100.0)
    return I_val, V_val, U_ocv

def sei_thermal_pde(x, y):
    T_delta = y[:, 0:1]
    c = y[:, 1:2]
    T_ref_point = x[:, 3:4] + 273.15
    T = T_delta * 50.0 + T_ref_point
    T_safe = torch.clamp(T, min=273.15, max=600.0)
    c_safe = torch.clamp(c, min=0.0, max=1.0)
    dT_t = dde.grad.jacobian(y, x, i=0, j=4) / t_max_seconds
    dc_t = dde.grad.jacobian(y, x, i=1, j=4) / t_max_seconds
    dT_xx = dde.grad.hessian(y, x, i=0, j=0)
    I, V, U_ocv = get_real_time_physics(x[:, 4:5])
    actual_A_sei = torch.pow(10.0, log_A_sei)  # log_A_sei populated below once restored
    joule_heat = torch.abs(I * (V - U_ocv))
    exponent = -Ea_sei / (R_gas * T_safe + 1e-6)
    rate = actual_A_sei * c_safe * torch.exp(torch.clamp(exponent, min=EXPONENT_CLAMP_MIN, max=EXPONENT_CLAMP_MAX))
    thermal_mass = (rho * Cp) + 1e-6
    res_thermal = (dT_t * 50.0) - (k_cond * dT_xx * 50.0 + joule_heat + H_sei * rate) / thermal_mass
    res_sei = dc_t + rate
    return [res_thermal, res_sei]

def mae_temp_metric(y_true, y_pred):
    if y_true is None:
        return torch.tensor(0.0)
    return torch.mean(torch.abs(y_true - y_pred[:, 0:1]))

def l2_temp_error(y_true, y_pred):
    if y_true is None:
        return torch.tensor(0.0)
    return torch.norm(y_true - y_pred[:, 0:1]) / (torch.norm(y_true) + 1e-7)

print(f"Reloaded {len(df)} rows from final_pinn_training_data to rebuild interpolation tables.")

In [0]:
# ---------------------------------------------------------
#  Reconstruct architecture using the real physics function, then restore
#  weights + calibrated parameters from the MLflow run specified above.
#
# NOTE: dde.data.TimePDE needs *a* PDE function to construct the Data object,
# but at inference time model.predict() only performs a forward pass through
# the trained network -- it does NOT re-solve or enforce the PDE residual.
# Physics constraints were baked into the weights during training only.
# The dummy geometry below does not need to match the training domain -- it's
# only used to construct the Data/Model objects, and no samples are ever
# drawn from it in this section (no model.train() call follows).
# ---------------------------------------------------------
geom = dde.geometry.Interval(0, 1)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)
data = dde.data.TimePDE(geomtime, sei_thermal_pde, [], num_domain=100)
# 5 Inputs: [Space, SoH, Re, Ambient, Time] -- matches the training architecture
net = dde.nn.FNN([5] + [64] * 4 + [2], "tanh", "Glorot normal")
net.float()
model = dde.model.Model(data, net)

# Download & restore
client = MlflowClient()
try:
    local_path = client.download_artifacts(run_id, artifact_path)
except Exception as e:
    raise RuntimeError(
        f"Could not download '{artifact_path}' from MLflow run '{run_id}'. "
        f"Double-check run_id in the cell above is correct and that this run "
        f"actually logged the checkpoint artifact in Section 6. "
        f"Original error: {e}"
    )

checkpoint = torch.load(local_path, map_location=torch.device('cpu'))

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.net.load_state_dict(checkpoint["model_state_dict"])

    # --- Fix: restore log_A_sei from checkpoint instead of using an
    # untrained default ---
    if "log_A_sei" in checkpoint:
        log_A_sei = dde.Variable(checkpoint["log_A_sei"])
        print(f"Restored A_sei = {10 ** checkpoint['log_A_sei']:.3e}")
    else:
        print("WARNING: no log_A_sei in checkpoint -- defaulting to 10.0 "
              "(likely far too low; see Section 4's diagnostic).")
        log_A_sei = dde.Variable(10.0)

    # --- Fix: restore the fitted OCV polynomial coefficients too ---
    if "ocv_coeffs" in checkpoint:
        ocv_coeffs = np.array(checkpoint["ocv_coeffs"])
        print("Restored fitted OCV polynomial coefficients.")
    else:
        raise RuntimeError(
            "No ocv_coeffs found in this checkpoint -- get_real_time_physics "
            "cannot evaluate the OCV curve without it. Retrain with the "
            "current Section 6 to produce a compatible checkpoint."
        )
else:
    raise RuntimeError(
        "Legacy checkpoint format detected (weights only, no log_A_sei/"
        "ocv_coeffs) -- this run predates the current checkpoint format and "
        "cannot be fully restored. Retrain with the current Section 6."
    )

model.compile(
    "adam",
    lr=1e-3,
    loss_weights=[10, 10, 30, 30, 20],  # kept consistent with the rebalanced training weights
    external_trainable_variables=log_A_sei,
    metrics=[mae_temp_metric, l2_temp_error]
)

print(f"Model restored from run_id={run_id}. Architecture: 5 inputs, 4 hidden layers, 2 outputs.")

In [0]:
# 4. DIGITAL TWIN SIMULATION
test_t = np.linspace(0, 1, 100)
s_soh, s_re, s_ambient = 0.8, 1.5, 24.0  # Simulate an 80% health battery at ~24C ambient

# Predict Internal (x=0) vs Surface (x=1)
core_in = np.column_stack((
    np.zeros(100),           # Space (x=0)
    np.full(100, s_soh),     # SoH
    np.full(100, s_re),      # Re
    np.full(100, s_ambient), # Ambient (Celsius) -- fix: new input dimension
    test_t                   # Time
)).astype(np.float32)

core_pred = model.predict(core_in)
print("Sample digital-twin prediction (first 5 rows, [T_delta, c]):")
print(core_pred[:5])

### Abuse Test — "Point of No Return" Simulation

Sweeps the **ambient** input from 25°C to 150°C at a fixed exposure time to find where the
predicted SEI stability `c` collapses, as a proxy for thermal runaway risk.

**Bonus fix:** the original cell computed this same 25→150°C ramp (`oven_t_c`/`oven_t_norm`) but
then built `abuse_test` using only the ordinary `[0,1]` time domain — the ramp was calculated and
never actually used, so the "oven test" was really just predicting along the normal time axis.
Now that ambient is a genuine input dimension (Section 5/6), the ramp is wired directly into the
model input, so this is a real ambient-stress sweep.

**v7 fixes:**
- **Extrapolation warning.** The network's `Ambient` input dimension was only ever trained on
  `[0, 45]°C` (`Hypercube(..., xmax=[..., 45.0])` in Section 5, matching the dataset's real
  ~4/24/43°C test groups). This sweep goes up to 150°C, so everything past 45°C ambient is the
  network extrapolating well outside its trained domain — not a physically grounded prediction. The
  plot below now visually separates the in-domain region from the extrapolated region instead of
  presenting the whole curve as equally trustworthy.
- **Data-driven danger threshold.** The previous `0.985` SEI-stability cutoff was an arbitrary
  constant. It's now derived from the model's own predictions at real, in-domain training
  conditions, so "danger" is defined relative to this model's real baseline behavior rather than a
  number carried over from before the Celsius/Kelvin unit fix.

**v8 fixes:**
- **The v7 threshold was methodologically circular.** Using the *1st percentile of the same
  distribution* as the danger cutoff is close to tautological — by definition, ~1% of any
  distribution sits at or below its own 1st percentile, regardless of whether those points mean
  anything physically. The actual run also showed `baseline_c.mean() = 1.0092` — **above 1.0**,
  which is impossible for a quantity meant to represent stability bounded in `[0,1]`. The network's
  raw `c` output isn't actually constrained to `[0,1]`; only the *internal* Arrhenius rate
  calculation clamps it (`c_safe`), not the value reported by `model.predict()`. The threshold is
  now: (1) clip `c` to `[0,1]` before computing any statistics, and (2) use a threshold based on how
  far below the *typical* baseline sits (mean − 3·std) rather than a percentile of the same data
  being evaluated.
- **DANGER trigger is now printed explicitly.** The previous version reported only the final
  verdict, leaving it ambiguous whether `core_temp` or `sei_health` (or both) actually crossed their
  threshold. Both conditions are now printed with their values against their thresholds.
- **Training-data density check** at the abuse test's fixed `SoH=0.8, Re=0.5` — satisfying the
  geometry's `[min,max]` bounds doesn't guarantee real training data density nearby; a DANGER
  verdict from a sparse, undertrained region of the input space isn't the same as one grounded in
  plenty of real examples.

In [0]:
# --- Baseline: in-domain SEI stability, for a data-driven danger threshold ---
# Uses the model's own predictions at the real recorded training conditions
# (not the raw NASA labels) so the baseline is on the same footing as the
# abuse-test predictions below -- both are model outputs, just at different
# (in-domain vs extrapolated) inputs.
baseline_points = np.column_stack((
    np.ones(len(df)),                   # Space (x=1, surface -- matches how observe_T was trained)
    df['SoH'].values / 100.0,           # SoH
    df['Re'].values * 10.0,             # Re
    df['ambient_temperature'].values,   # Ambient (Celsius) -- real, in-domain values only
    df['Time_normalized'].values        # Time
)).astype(np.float32)

baseline_pred = model.predict(baseline_points)

# v8 fix: clip to the physically valid [0,1] range before computing stats.
# The raw network output isn't constrained to [0,1] -- only the *internal*
# c_safe used inside the Arrhenius rate calculation is clamped; the value
# model.predict() reports is not, which is why the previous run showed
# baseline_c.mean() = 1.0092 (impossible for a bounded stability fraction).
baseline_c = np.clip(baseline_pred[:, 1], 0.0, 1.0)

baseline_mean = baseline_c.mean()
baseline_std = baseline_c.std()

print("In-domain SEI stability (c) baseline, from real training conditions (clipped to [0,1]):")
print(f"  min={baseline_c.min():.4f}, mean={baseline_mean:.4f}, std={baseline_std:.4f}, "
      f"p1={np.percentile(baseline_c, 1):.4f}, p5={np.percentile(baseline_c, 5):.4f}")

# v8 fix: threshold based on how far below the *typical* baseline a reading
# falls (mean - 3*std), rather than a percentile of the same distribution
# being evaluated -- a percentile is close to tautological (~1% of any
# distribution sits at or below its own 1st percentile by construction).
# This is still a heuristic derived from the model's own outputs, not a
# literature-grounded physical failure point -- treat it as "notably below
# this model's typical baseline," not a validated safety limit.
SEI_DANGER_THRESHOLD = float(max(baseline_mean - 3 * baseline_std, 0.0))
print(f"\nUsing data-driven danger threshold (baseline mean - 3*std): {SEI_DANGER_THRESHOLD:.4f}")
print("(Still a heuristic based on this model's own in-domain outputs, not a literature-validated "
      "physical failure point -- useful as a relative 'notably abnormal' flag, not a certified limit.)")

In [0]:
# --- Training-data density check at the abuse-test's fixed SoH/Re ---
# Satisfying the geometry's [min,max] bounds doesn't guarantee real training
# data density nearby -- a DANGER verdict from a sparse, undertrained corner
# of the input space isn't the same as one grounded in plenty of real
# examples. Checks how well-represented (SoH~0.8, Re~0.5) -- the abuse test's
# fixed conditions -- actually are in the training data.
soh_target, re_target = 0.8, 0.5
soh_tol, re_tol = 0.05, 0.1

nearby = df[
    (np.abs(df['SoH'].values / 100.0 - soh_target) < soh_tol)
    & (np.abs(df['Re'].values * 10.0 - re_target) < re_tol)
]

print(f"Training rows near SoH~{soh_target*100:.0f}%, Re~{re_target} "
      f"(+/-{soh_tol*100:.0f}%, +/-{re_tol}) -- the abuse test's fixed conditions:")
print(f"  {len(nearby)} / {len(df)} rows ({100*len(nearby)/len(df):.1f}%)")

if len(nearby) < 50:
    print("WARNING: sparse training coverage near these conditions -- any DANGER verdict at these "
          "SoH/Re values may reflect an undertrained region of the input space, not necessarily "
          "real physics. Consider re-running the abuse test at SoH/Re values with denser coverage, "
          "or treat this specific verdict with extra caution.")
else:
    print("Reasonable coverage near these conditions.")

In [0]:
# --- Simulation of Ignition Threshold (ambient stress sweep) ---
# Reminder: this is model.predict() -- a forward pass through the trained
# network -- not a re-solved PDE. The physics is encoded in the learned weights.
oven_ambient_c = np.linspace(25, 150, 100)  # Ambient stress sweep: 25C -> 150C

# Trained domain boundary for the Ambient input (Section 5's Hypercube xmax=45.0)
AMBIENT_TRAINED_MAX_C = 45.0
in_domain_mask = oven_ambient_c <= AMBIENT_TRAINED_MAX_C

if not in_domain_mask.all():
    n_extrap = (~in_domain_mask).sum()
    print(f"WARNING: {n_extrap}/{len(oven_ambient_c)} sweep points exceed the trained Ambient "
          f"domain (0-{AMBIENT_TRAINED_MAX_C:.0f}C). Predictions there are extrapolation, not a "
          f"physically grounded result -- treat that portion of the plot as illustrative only.")

# Input: [x=0 (core), SoH=0.8, Re=0.5, Ambient=sweep, t=1.0 (full exposure)]
abuse_test = np.column_stack((
    np.zeros(100),               # Core
    np.full(100, 0.8),           # 80% SoH
    np.full(100, 0.5),           # Standard Re
    oven_ambient_c,              # Ambient stress sweep -- fix: now actually used
    np.full(100, 1.0)            # Time: full exposure (t_norm = 1.0)
)).astype(np.float32)

res = model.predict(abuse_test)
# T = T_delta*50 + T_ref_ambient(K); in Celsius this simplifies to T_delta*50 + ambient_C
core_temp = res[:, 0] * 50 + oven_ambient_c
# v8 fix: clip to [0,1] -- see the baseline cell above for why the raw
# network output isn't already bounded there.
sei_health = np.clip(res[:, 1], 0.0, 1.0)

# Status logic restricted to the in-domain portion of the sweep -- an
# extrapolated reading shouldn't be allowed to (mis)inform a safety verdict.
core_temp_in_domain = core_temp[in_domain_mask]
sei_health_in_domain = sei_health[in_domain_mask]

CORE_TEMP_DANGER_C = 80.0
CORE_TEMP_WARNING_C = 60.0

temp_triggered = core_temp_in_domain.max() > CORE_TEMP_DANGER_C
sei_triggered = sei_health_in_domain.min() < SEI_DANGER_THRESHOLD

# v8 fix: print explicitly which condition (if any) triggered the verdict,
# instead of only reporting the final status -- the previous version left it
# ambiguous whether core_temp or sei_health (or both) actually crossed
# their threshold.
print(f"core_temp_in_domain.max() = {core_temp_in_domain.max():.2f}C "
      f"(threshold: {CORE_TEMP_DANGER_C:.0f}C) -> {'EXCEEDED' if temp_triggered else 'ok'}")
print(f"sei_health_in_domain.min() = {sei_health_in_domain.min():.4f} "
      f"(threshold: {SEI_DANGER_THRESHOLD:.4f}) -> {'BELOW THRESHOLD' if sei_triggered else 'ok'}")

if temp_triggered or sei_triggered:
    status = "DANGER: THERMAL RUNAWAY PREDICTED"
elif core_temp_in_domain.max() > CORE_TEMP_WARNING_C:
    status = "WARNING: ACCELERATED AGING DETECTED"
else:
    status = "SAFE"

print(f"\nStatus (evaluated on in-domain sweep points only, ambient <= {AMBIENT_TRAINED_MAX_C:.0f}C): {status}")

# Visualisation -- in-domain vs extrapolated portions are visually distinct
plt.figure(figsize=(10, 5))
plt.plot(core_temp[in_domain_mask], sei_health[in_domain_mask], 'g-', linewidth=2,
         label='SEI Health (in-domain, Ambient <= 45C)')
if not in_domain_mask.all():
    # Include the boundary point so the two segments connect visually
    boundary_idx = np.argmax(~in_domain_mask)
    extrap_idx = slice(max(boundary_idx - 1, 0), None)
    plt.plot(core_temp[extrap_idx], sei_health[extrap_idx], 'g--', alpha=0.4, linewidth=2,
             label='SEI Health (EXTRAPOLATED, Ambient > 45C -- not physically grounded)')
    plt.axvline(x=core_temp[boundary_idx], color='gray', linestyle=':', alpha=0.7,
                label=f'Trained-domain boundary (Ambient={AMBIENT_TRAINED_MAX_C:.0f}C)')

plt.axvline(x=80, color='r', linestyle='--', label='Critical Threshold (80C)')
plt.axhline(y=SEI_DANGER_THRESHOLD, color='orange', linestyle=':',
            label=f'Chemical Failure Limit (data-driven, {SEI_DANGER_THRESHOLD:.3f})')
plt.xlabel('Internal Temperature (degC)')
plt.ylabel('Chemical Stability (c)')
plt.title('Digital Twin: Predicting the Point of No Return (Ambient Stress Sweep)')
plt.legend()
plt.show()